In [1]:
from google.colab import files
uploaded = files.upload()

Saving colab_upload.zip to colab_upload.zip


In [2]:
from google.colab import files
uploaded = files.upload()

Saving data_folder.zip to data_folder.zip


In [3]:
import zipfile, os

with zipfile.ZipFile("colab_upload.zip", "r") as z:
    z.extractall("fin-research-agent")

%cd fin-research-agent

with zipfile.ZipFile("../data_folder.zip", "r") as z:
    z.extractall(".")

/content/fin-research-agent


In [4]:
import shutil

for fname in os.listdir("."):
    if "\\" in fname:
        proper_path = fname.replace("\\", "/")
        os.makedirs(os.path.dirname(proper_path), exist_ok=True)
        shutil.move(fname, proper_path)

!find . -name "*.py" | sort

./agents/analyst_agent.py
./agents/data_agent.py
./agents/graph.py
./agents/judge_agent.py
./agents/llm_client.py
./agents/prompts/analyst_prompt.py
./agents/prompts/judge_prompt.py
./agents/report_agent.py
./agents/retriever_agent.py
./data/cache.py
./data/edgar_fetcher.py
./data/market_data.py
./eval/baseline_report.py
./eval/day30_comparison.py
./eval/ragas_eval.py
./retrieval/chunker.py
./retrieval/confidence_scorer.py
./retrieval/embedder.py
./retrieval/vector_store.py


In [5]:
for folder in ["agents", "agents/prompts", "eval", "retrieval", "data"]:
    init_path = os.path.join(folder, "__init__.py")
    if not os.path.exists(init_path):
        open(init_path, "w").close()

!find . -name "__init__.py"

./data/__init__.py
./retrieval/__init__.py
./agents/__init__.py
./agents/prompts/__init__.py
./eval/__init__.py


In [6]:
!pip install -r requirements.txt
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 125.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2

In [7]:
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
!cat agents/llm_client.py

"""
agents/llm_client.py

Shared LLM client for all agents in the pipeline.
Single place to configure model, API key, timeouts, and retries.
"""
import os
import httpx
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

_http_client = httpx.Client(timeout=120.0)

_llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    openai_api_key=os.getenv("NVIDIA_API_KEY"),
    openai_api_base="https://integrate.api.nvidia.com/v1",
    temperature=1.0, top_p=1.0, max_tokens=4096,
    timeout=120.0, max_retries=0,
)
llm = _llm  # public alias for use with .with_structured_output() elsewhere

def call_llm(prompt: str) -> str:
    """Send a prompt to the configured LLM and return the response string."""
    response = _llm.invoke(prompt)
    return response.content if hasattr(response, "content") else str(response)


In [9]:
patched = '''"""
agents/llm_client.py
Shared LLM client for all agents in the pipeline.
Single place to configure model, API key, timeouts, and retries.
"""
import os
import time
import httpx
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

_http_client = httpx.Client(timeout=180.0)

_llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    openai_api_key=os.getenv("NVIDIA_API_KEY"),
    openai_api_base="https://integrate.api.nvidia.com/v1",
    temperature=1.0, top_p=1.0, max_tokens=4096,
    timeout=180.0,
    max_retries=0,  # we handle retries manually below
)

llm = _llm  # public alias for use with .with_structured_output() elsewhere


def call_llm(prompt: str, max_attempts: int = 3, wait_seconds: int = 10) -> str:
    """Send a prompt to the configured LLM and return the response string.
    Retries up to max_attempts times on timeout, with wait_seconds between tries."""
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            response = _llm.invoke(prompt)
            return response.content if hasattr(response, "content") else str(response)
        except Exception as e:
            last_error = e
            print(f"[call_llm] Attempt {attempt}/{max_attempts} failed: {type(e).__name__}. "
                  f"{'Retrying in ' + str(wait_seconds) + 's...' if attempt < max_attempts else 'Giving up.'}")
            if attempt < max_attempts:
                time.sleep(wait_seconds)
    raise last_error
'''

with open("agents/llm_client.py", "w") as f:
    f.write(patched)

print("agents/llm_client.py patched successfully.")
!cat agents/llm_client.py

agents/llm_client.py patched successfully.
"""
agents/llm_client.py
Shared LLM client for all agents in the pipeline.
Single place to configure model, API key, timeouts, and retries.
"""
import os
import time
import httpx
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

_http_client = httpx.Client(timeout=180.0)

_llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    openai_api_key=os.getenv("NVIDIA_API_KEY"),
    openai_api_base="https://integrate.api.nvidia.com/v1",
    temperature=1.0, top_p=1.0, max_tokens=4096,
    timeout=180.0,
    max_retries=0,  # we handle retries manually below
)

llm = _llm  # public alias for use with .with_structured_output() elsewhere


def call_llm(prompt: str, max_attempts: int = 3, wait_seconds: int = 10) -> str:
    """Send a prompt to the configured LLM and return the response string.
    Retries up to max_attempts times on timeout, with wait_seconds between tries."""
    last_error = None
    for attempt in ran

In [10]:
patched = '''"""
agents/llm_client.py
Shared LLM client for all agents in the pipeline.
Single place to configure model, API key, timeouts, and retries.
"""
import os
import time
import httpx
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

_http_client = httpx.Client(timeout=180.0)

_llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    openai_api_key=os.getenv("NVIDIA_API_KEY"),
    openai_api_base="https://integrate.api.nvidia.com/v1",
    temperature=1.0, top_p=1.0, max_tokens=4096,
    timeout=180.0,
    max_retries=0,  # we handle retries manually below
)

llm = _llm  # public alias for use with .with_structured_output() elsewhere


def call_llm(prompt: str, max_attempts: int = 3, wait_seconds: int = 10) -> str:
    """Send a prompt to the configured LLM and return the response string.
    Retries up to max_attempts times on timeout, with wait_seconds between tries."""
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            response = _llm.invoke(prompt)
            return response.content if hasattr(response, "content") else str(response)
        except Exception as e:
            last_error = e
            print(f"[call_llm] Attempt {attempt}/{max_attempts} failed: {type(e).__name__}. "
                  f"{'Retrying in ' + str(wait_seconds) + 's...' if attempt < max_attempts else 'Giving up.'}")
            if attempt < max_attempts:
                time.sleep(wait_seconds)
    raise last_error
'''

with open("agents/llm_client.py", "w") as f:
    f.write(patched)

print("agents/llm_client.py patched successfully.")
!cat agents/llm_client.py

agents/llm_client.py patched successfully.
"""
agents/llm_client.py
Shared LLM client for all agents in the pipeline.
Single place to configure model, API key, timeouts, and retries.
"""
import os
import time
import httpx
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

_http_client = httpx.Client(timeout=180.0)

_llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    openai_api_key=os.getenv("NVIDIA_API_KEY"),
    openai_api_base="https://integrate.api.nvidia.com/v1",
    temperature=1.0, top_p=1.0, max_tokens=4096,
    timeout=180.0,
    max_retries=0,  # we handle retries manually below
)

llm = _llm  # public alias for use with .with_structured_output() elsewhere


def call_llm(prompt: str, max_attempts: int = 3, wait_seconds: int = 10) -> str:
    """Send a prompt to the configured LLM and return the response string.
    Retries up to max_attempts times on timeout, with wait_seconds between tries."""
    last_error = None
    for attempt in ran

In [12]:
import os
os.environ["JUDGE_MODE"] = "api"

!python -m eval.day30_comparison

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
modules.json: 100% 349/349 [00:00<00:00, 1.22MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 662kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 29.6MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 319kB/s]
config.json: 100% 612/612 [00:00<00:00, 3.05MB/s]

model.safetensors: downloading bytes:  94% 85.0M/90.9M [00:01<00:00, 103MB/s, 7.45MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:01<00:00, 73.2MB/s, 8.11MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 78.2MB/s, 8.83MB/s  ]
Loading weights: 100% 103/

In [14]:
content = open("agents/judge_agent.py").read()
patched = content.replace(
    'model_name="finetune/full_adapter_v1"',
    'model_name=os.getenv("FINETUNED_ADAPTER_PATH", "finetune/full_adapter_v1")'
)
open("agents/judge_agent.py", "w").write(patched)
print("Patched. Verifying:")
!grep "FINETUNED_ADAPTER_PATH" agents/judge_agent.py

Patched. Verifying:
            model_name=os.getenv("FINETUNED_ADAPTER_PATH", "finetune/full_adapter_v1"),


In [17]:
import os, sys
sys.path.insert(0, ".")
os.environ["FINETUNED_ADAPTER_PATH"] = "full_adapter_v1"

from agents.judge_agent import _load_finetuned_model
from agents.prompts.judge_prompt import build_judge_prompt

# Fixed — chunks as dicts matching what your pipeline actually produces
test_report = {
    "bull_points": ["Revenue grew 10% (Section: Item 7)"],
    "bear_points": ["Competition is increasing (Section: Item 1A)"],
    "summary": "Mixed results overall.",
    "citations": ["Item 7", "Item 1A"]
}
test_chunks = [
    {"section": "Item 7", "text": "Revenue increased 10% year over year driven by cloud services."},
    {"section": "Item 1A", "text": "Competition in the technology sector remains intense."}
]
prompt = build_judge_prompt(test_report, test_chunks)

# Load model (already cached if adapter was loaded earlier this session)
model, tokenizer = _load_finetuned_model()

# Generate and print RAW output
formatted = f"### Instruction:\n{prompt}\n\n### Response:\n"
inputs = tokenizer(formatted, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=300, use_cache=True)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
raw_response = generated_text.split("### Response:\n")[-1]

print("===== RAW FINE-TUNED OUTPUT =====")
print(repr(raw_response))
print("===== END =====")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.9.4 patched 36 layers with 36 QKV layers, 36 O layers and 0 MLP layers.
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== RAW FINE-TUNED OUTPUT =====
"Flagged Issues:\n- Bull point 1: Grounding is ungrounded. Source chunk [Section: Item 7] does not provide a specific reason for revenue growth. It only states that revenue increased 10% year over year, without any additional information about the cause.\n- Bear point 1: Grounding is ungrounded. Source chunk [Section: Item 1A] mentions competition in the technology sector, but it does not specify that competition is increasing. The statement could mean that competition has remained steady, which would not necessarily contradict the source chunk.\n\nflagged_issues = ['Bull point 1: Grounding is ungrounded. Source chunk [Section: Item 7] does not provide a specific reason for revenue growth. It only states that revenue increased 10% year over year, without any additional information about the cause.', 'Bear point 1: Grounding is ungrounded. Source chunk [Section: Item 1A] mentions competition in the technology sector, but it does not specify that competi

In [18]:
content = open("agents/judge_agent.py").read()

old_extraction = '''        match = re.search(r"\\{.*?\\}", raw_response, re.DOTALL)
        if not match:
            raise ValueError("No JSON object found in fine-tuned output")

        parsed = json.loads(match.group(0))
        validated = JudgeScore(**parsed)
        return {"judge_score": validated.model_dump()}'''

new_extraction = '''        # Try clean JSON first
        match = re.search(r"\\{.*?\\}", raw_response, re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group(0))
                validated = JudgeScore(**parsed)
                return {"judge_score": validated.model_dump()}
            except Exception:
                pass  # fall through to prose extraction

        # Prose extraction — model outputs "Grounding: 7" style lines
        def extract_score(pattern):
            m = re.search(pattern, raw_response, re.IGNORECASE)
            return max(1, min(10, int(m.group(1)))) if m else 5

        grounding    = extract_score(r"Grounding[:\\s]+([0-9]+)")
        completeness = extract_score(r"Completeness[:\\s]+([0-9]+)")
        clarity      = extract_score(r"Clarity[:\\s]+([0-9]+)")
        overall      = extract_score(r"Overall[:\\s]+([0-9]+)")
        if overall == 5:  # wasn\'t found, compute it
            overall = max(1, min(10, round((grounding + completeness + clarity) / 3)))
            if min(grounding, completeness, clarity) <= 3:
                overall = min(overall, 5)

        # Extract flagged_issues list if present
        issues = []
        list_match = re.search(r"flagged_issues\\s*=\\s*\\[(.+?)\\]", raw_response, re.DOTALL)
        if list_match:
            raw_list = list_match.group(1)
            issues = [s.strip().strip("\'\\\"") for s in re.split(r",\\s*\'|,\\s*\\"", raw_list) if s.strip().strip("\'\\\"")]
        else:
            # Fall back to bullet-point style flagged issues
            bullet_matches = re.findall(r"[-•]\\s*(.+?)(?=\\n[-•]|\\nGrounding|\\nCompleteness|\\nClarity|\\nOverall|$)", raw_response, re.DOTALL)
            issues = [m.strip() for m in bullet_matches if len(m.strip()) > 10][:5]

        validated = JudgeScore(
            grounding=grounding,
            completeness=completeness,
            clarity=clarity,
            overall=overall,
            flagged_issues=issues
        )
        return {"judge_score": validated.model_dump()}'''

patched = content.replace(old_extraction, new_extraction)

if old_extraction in content:
    with open("agents/judge_agent.py", "w") as f:
        f.write(patched)
    print("Patched successfully.")
else:
    print("OLD STRING NOT FOUND — paste agents/judge_agent.py content so I can fix manually.")

Patched successfully.


In [19]:
import os
os.environ["JUDGE_MODE"] = "finetuned"
os.environ["FINETUNED_ADAPTER_PATH"] = "full_adapter_v1"

!python -m eval.day30_comparison

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Loading weights: 100% 103/103 [00:00<00:00, 22807.16it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status  

In [20]:
# In Colab — print the final file so you can copy it
!cat agents/judge_agent.py

import os
import re
import json

from pydantic import BaseModel
from agents.llm_client import llm
from agents.prompts.judge_prompt import build_judge_prompt


class JudgeScore(BaseModel):
    grounding: int
    completeness: int
    clarity: int
    overall: int
    flagged_issues: list[str]


FALLBACK_JUDGE_SCORE = {
    "grounding": -1,
    "completeness": -1,
    "clarity": -1,
    "overall": -1,
    "flagged_issues": ["Judge evaluation failed — LLM call error, this score is not real"],
}

JUDGE_MODE = os.getenv("JUDGE_MODE", "api")  # "api" or "finetuned"

_finetuned_model = None
_finetuned_tokenizer = None


def _load_finetuned_model():
    global _finetuned_model, _finetuned_tokenizer
    if _finetuned_model is None:
        from unsloth import FastLanguageModel
        _finetuned_model, _finetuned_tokenizer = FastLanguageModel.from_pretrained(
            model_name=os.getenv("FINETUNED_ADAPTER_PATH", "finetune/full_adapter_v1"),
            max_seq_length=6144,
            load